# BirdCLEF 2026 — Perch Embedding Precompute v3 (GCP VM, ONNX)
## Uses `perch_v2_cpu.onnx` via onnxruntime — **no TensorFlow**
## Computes TRAIN embeddings only — test audio is not available on GCP
## Run this BEFORE birdclef2026-train-v23-perch-gru.ipynb

### What changed vs v2:
- **No TF**: uses `perch_v2_cpu.onnx` from `birdclef2026-perch-to-onnx-gcp.ipynb`
- **Output dataset**: `birdclef-2026-perch-embs-v3`
- **Sanity check**: verifies embedding is non-trivial (std > 0.1) after first batch
- Embedding extraction logic identical to v2 (focal clips + soundscape windows)

### Quick start on GCP VM:
```bash
export BIRDCLEF_DATA=~/data/birdclef-2026
export PERCH_ONNX=~/birdclef-output/perch_v2_cpu.onnx   # from perch-to-onnx-gcp.ipynb
export BIRDCLEF_OUT=~/birdclef-output
```
Then run all cells. Upload output as `birdclef-2026-perch-embs-v3`.

### Prerequisites:
1. Run `birdclef2026-perch-to-onnx-gcp.ipynb` first to produce `perch_v2_cpu.onnx`
2. Set `PERCH_ONNX` env var to point to that file

In [ ]:
# === CELL 1: INSTALL & IMPORTS ===
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'onnxruntime', 'soundfile', 'librosa', 'tqdm'],
    check=False,
)

import os, gc, json
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import onnxruntime as ort
from tqdm import tqdm

PERCH_SR      = 32000
SECONDS       = 5
PERCH_TARGET  = PERCH_SR * SECONDS   # 160,000 samples
_PERCH_BATCH  = 32
_MODEL_LABEL  = 'perch_v2_cpu_onnx'  # embedded in verification output

print(f'onnxruntime {ort.__version__}')
print(f'Providers: {ort.get_available_providers()}')

In [ ]:
# === CELL 2: PATHS ===
def _first_existing(*candidates):
    return next(
        (str(Path(p).expanduser()) for p in candidates
         if Path(p).expanduser().exists()),
        str(Path(candidates[0]).expanduser()),
    )

_data_root  = os.environ.get('BIRDCLEF_DATA', '~/data/birdclef-2026')
_out_root   = os.environ.get('BIRDCLEF_OUT',  '~/birdclef-output')
_onnx_env   = os.environ.get('PERCH_ONNX',   '~/birdclef-output/perch_v2_cpu.onnx')

TRAIN_META_CSV = _first_existing(
    f'{_data_root}/train.csv',
    f'{_data_root}/birdclef-2026/train.csv',
)
TRAIN_AUDIO_DIR = _first_existing(
    f'{_data_root}/train_audio',
    f'{_data_root}/birdclef-2026/train_audio',
)
SOUNDSCAPE_DIR = _first_existing(
    f'{_data_root}/train_soundscapes',
    f'{_data_root}/birdclef-2026/train_soundscapes',
)
SOUNDSCAPE_LABELS_CSV = _first_existing(
    f'{_data_root}/train_soundscapes_labels.csv',
    f'{_data_root}/birdclef-2026/train_soundscapes_labels.csv',
)

# ONNX model (output of birdclef2026-perch-to-onnx-gcp.ipynb)
_onnx_candidates = [
    _onnx_env,
    os.path.expanduser('~/birdclef-output/perch_v2_cpu.onnx'),
    os.path.expanduser('~/perch_v2_cpu.onnx'),
    '/opt/perch_v2_cpu.onnx',
]
ONNX_PATH = None
for _cand in _onnx_candidates:
    _p = Path(_cand).expanduser()
    print(f'  Checking {_p}  exists={_p.exists()}')
    if _p.exists():
        ONNX_PATH = str(_p)
        print(f'  => SELECTED: {ONNX_PATH}')
        break

if ONNX_PATH is None:
    raise RuntimeError(
        'perch_v2_cpu.onnx not found.\n'
        'Run birdclef2026-perch-to-onnx-gcp.ipynb first to produce the ONNX file,\n'
        'then set PERCH_ONNX=~/path/to/perch_v2_cpu.onnx and re-run.'
    )

EMBD_DIR = Path(_out_root).expanduser() / 'perch_embeddings_v3'
EMBD_DIR.mkdir(parents=True, exist_ok=True)

df    = pd.read_csv(TRAIN_META_CSV)
sc_df = pd.read_csv(SOUNDSCAPE_LABELS_CSV)

print(f'\nTraining clips    : {len(df)}')
print(f'Soundscape rows   : {len(sc_df)}')
print(f'TRAIN_AUDIO_DIR   : {TRAIN_AUDIO_DIR}  (exists={os.path.exists(TRAIN_AUDIO_DIR)})')
print(f'SOUNDSCAPE_DIR    : {SOUNDSCAPE_DIR}  (exists={os.path.exists(SOUNDSCAPE_DIR)})')
print(f'ONNX_PATH         : {ONNX_PATH}')
print(f'EMBD_DIR          : {EMBD_DIR}')
print(f'Already cached    : {len(list(EMBD_DIR.glob("*.npy")))} embeddings')

In [ ]:
# === CELL 3: LOAD ONNX SESSION + SANITY CHECK ===
print(f'Loading ONNX session from {ONNX_PATH} ...')

_sess_opts = ort.SessionOptions()
_sess_opts.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
_sess_opts.intra_op_num_threads = os.cpu_count() or 8
_ort_session = ort.InferenceSession(
    ONNX_PATH,
    sess_options=_sess_opts,
    providers=['CPUExecutionProvider'],
)
_onnx_inp = _ort_session.get_inputs()[0].name

# Auto-detect 1536-d embedding output
_onnx_emb_key = None
for _out in _ort_session.get_outputs():
    if _out.shape and _out.shape[-1] == 1536:
        _onnx_emb_key = _out.name
        break

# Check model_info.json if present (written by perch-to-onnx-gcp.ipynb)
_info_json = Path(ONNX_PATH).parent / 'model_info.json'
if _info_json.exists():
    _info = json.load(open(_info_json))
    _onnx_inp     = _info.get('input_name', _onnx_inp)
    _onnx_emb_key = _info.get('embedding_output_name', _onnx_emb_key)
    print(f'Loaded model_info.json: input={_onnx_inp!r}, emb={_onnx_emb_key!r}')

if _onnx_emb_key is None:
    # Runtime detection via a dummy pass
    _dummy = np.zeros((1, PERCH_TARGET), dtype=np.float32)
    _dummy_outs = _ort_session.run(None, {_onnx_inp: _dummy})
    _out_ns = [o.name for o in _ort_session.get_outputs()]
    for _nm, _ov in zip(_out_ns, _dummy_outs):
        if _ov.ndim >= 2 and _ov.shape[-1] == 1536:
            _onnx_emb_key = _nm; break
    assert _onnx_emb_key, 'Cannot find 1536-d output in ONNX model.'

_out_names = [o.name for o in _ort_session.get_outputs()]
_emb_idx   = _out_names.index(_onnx_emb_key)
print('Running sanity check ...')

_test_audio = np.sin(2 * np.pi * 440 * np.linspace(0, 5, PERCH_TARGET)).astype(np.float32)[None]
_test_outs  = _ort_session.run(None, {_onnx_inp: _test_audio})
_test_emb   = _test_outs[_emb_idx]
if _test_emb.ndim == 3:
    _test_emb = _test_emb.mean(axis=1)
_test_emb = _test_emb[0]

assert _test_emb.shape == (1536,), f'Expected (1536,) but got {_test_emb.shape}'
assert _test_emb.std() > 0.1, f'Embedding std={_test_emb.std():.4f} near zero — ONNX model may be corrupt.'
print(f'\u2705 Sanity check passed: shape={_test_emb.shape}, mean={_test_emb.mean():.4f}, std={_test_emb.std():.4f}')
print(f'   input={_onnx_inp!r}  emb_output={_onnx_emb_key!r}')
del _test_audio, _test_outs, _test_emb

In [ ]:
# === CELL 4: FOCAL CLIP EMBEDDINGS ===
assert os.path.exists(TRAIN_AUDIO_DIR), f'Training audio not found at {TRAIN_AUDIO_DIR}'

print('Extracting focal clip embeddings ...')

_audio_batch, _name_batch = [], []

def _flush_batch():
    batch = np.stack(_audio_batch)  # (B, 160000)
    outs  = _ort_session.run(None, {_onnx_inp: batch})
    embs  = outs[_emb_idx]         # (B, 1536) or (B, T, 1536)
    if embs.ndim == 3:
        embs = embs.mean(axis=1)
    for name, emb in zip(_name_batch, embs):
        np.save(str(EMBD_DIR / name), emb.astype(np.float32))
    _audio_batch.clear()
    _name_batch.clear()

failed  = 0
skipped = 0

for _, row in tqdm(df.iterrows(), total=len(df), desc='Focal embed'):
    audio_path = Path(TRAIN_AUDIO_DIR) / row['filename']
    emb_name   = row['filename'].replace('/', '_') + '.npy'
    emb_path   = EMBD_DIR / emb_name

    if emb_path.exists():
        skipped += 1
        continue

    try:
        _info           = sf.info(str(audio_path))
        _frames_to_read = int(PERCH_TARGET * (_info.samplerate / PERCH_SR)) + 4096
        y, sr0          = sf.read(str(audio_path), frames=_frames_to_read, always_2d=False)
        if y.ndim == 2:
            y = y.mean(axis=1)
        if sr0 != PERCH_SR:
            y = librosa.resample(y.astype(np.float32), orig_sr=sr0, target_sr=PERCH_SR)
        y = y.astype(np.float32)
        if len(y) < PERCH_TARGET:
            y = np.pad(y, (0, PERCH_TARGET - len(y)))
        else:
            y = y[:PERCH_TARGET]
        _audio_batch.append(y)
        _name_batch.append(emb_name)
        if len(_audio_batch) >= _PERCH_BATCH:
            _flush_batch()
    except Exception as e:
        failed += 1
        if failed <= 5:
            print(f'  WARN focal failed: {row["filename"]} — {e}')

if _audio_batch:
    _flush_batch()

focal_saved = len([f for f in EMBD_DIR.glob('*.npy') if not f.stem.startswith('soundscape_')])
print(f'\nFocal done! saved={focal_saved}  skipped={skipped}  failed={failed}')

In [ ]:
# === CELL 5: SOUNDSCAPE WINDOW EMBEDDINGS ===
# Identical windowing to inference: y[end_samp - PERCH_TARGET : end_samp]
# Naming: soundscape_{stem}_{end_secs}s.npy
assert os.path.exists(SOUNDSCAPE_DIR), f'Soundscape dir not found: {SOUNDSCAPE_DIR}'

def _parse_hms_to_secs(s: str) -> int:
    parts = str(s).strip().split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])

_sc_wave_cache = {}  # stem -> y_float32 @32kHz
_sc_audio_batch, _sc_name_batch = [], []

def _flush_sc_batch():
    if not _sc_audio_batch:
        return
    batch = np.stack(_sc_audio_batch)  # (B, 160000)
    outs  = _ort_session.run(None, {_onnx_inp: batch})
    embs  = outs[_emb_idx]            # (B, 1536) or (B, T, 1536)
    if embs.ndim == 3:
        embs = embs.mean(axis=1)
    for name, emb in zip(_sc_name_batch, embs):
        np.save(str(EMBD_DIR / name), emb.astype(np.float32))
    _sc_audio_batch.clear()
    _sc_name_batch.clear()

failed_sc  = 0
skipped_sc = 0

sc_df_sorted = sc_df.sort_values('filename').reset_index(drop=True)

for _, row in tqdm(sc_df_sorted.iterrows(), total=len(sc_df_sorted), desc='Soundscape embed'):
    stem     = Path(str(row['filename'])).stem
    end_secs = _parse_hms_to_secs(row['end'])
    emb_name = f'soundscape_{stem}_{end_secs}s.npy'
    emb_path = EMBD_DIR / emb_name

    if emb_path.exists():
        skipped_sc += 1
        continue

    try:
        if stem not in _sc_wave_cache:
            audio_path = None
            for _ext in ['.ogg', '.wav', '.flac']:
                _cand = Path(SOUNDSCAPE_DIR) / f'{stem}{_ext}'
                if _cand.exists():
                    audio_path = _cand
                    break
            if audio_path is None:
                if failed_sc <= 3:
                    print(f'  WARN: soundscape audio not found: {stem}')
                failed_sc += 1
                continue
            y_full, sr0 = sf.read(str(audio_path), always_2d=False)
            if y_full.ndim == 2:
                y_full = y_full.mean(axis=1)
            if sr0 != PERCH_SR:
                y_full = librosa.resample(y_full.astype(np.float32), orig_sr=sr0, target_sr=PERCH_SR)
            _sc_wave_cache[stem] = y_full.astype(np.float32)
            if len(_sc_wave_cache) > 5:
                oldest = next(iter(_sc_wave_cache))
                del _sc_wave_cache[oldest]

        y_full     = _sc_wave_cache[stem]
        end_samp   = int(end_secs * PERCH_SR)
        start_samp = max(0, end_samp - PERCH_TARGET)
        clip       = y_full[start_samp:end_samp]
        if len(clip) < PERCH_TARGET:
            clip = np.pad(clip, (0, PERCH_TARGET - len(clip)))

        _sc_audio_batch.append(clip)
        _sc_name_batch.append(emb_name)
        if len(_sc_audio_batch) >= _PERCH_BATCH:
            _flush_sc_batch()

    except Exception as e:
        failed_sc += 1
        if failed_sc <= 5:
            print(f'  WARN soundscape failed: {stem} @ {end_secs}s — {e}')

if _sc_audio_batch:
    _flush_sc_batch()

del _sc_wave_cache
gc.collect()

soundscape_saved = len([f for f in EMBD_DIR.glob('soundscape_*.npy')])
print(f'\nSoundscape done! saved={soundscape_saved}  skipped={skipped_sc}  failed={failed_sc}')

del _ort_session
gc.collect()
print('ONNX session released.')

In [ ]:
# === CELL 6: VERIFY ===
all_emb_files = list(EMBD_DIR.glob('*.npy'))
focal_files   = [f for f in all_emb_files if not f.stem.startswith('soundscape_')]
sc_files      = [f for f in all_emb_files if f.stem.startswith('soundscape_')]

print(f'Total embeddings      : {len(all_emb_files)}')
print(f'  Focal clip embeds   : {len(focal_files)}')
print(f'  Soundscape embeds   : {len(sc_files)}')

# Check a few random files
import random
for _sample_f in random.sample(all_emb_files, min(5, len(all_emb_files))):
    _emb = np.load(str(_sample_f))
    assert _emb.shape == (1536,), f'Bad shape {_emb.shape} in {_sample_f.name}'
    assert _emb.std() > 0.05, (
        f'Near-zero embedding in {_sample_f.name} (std={_emb.std():.4f}) — '
        f'check model loaded correctly'
    )

print(f'✅ Shape+std check passed for sampled files')
print(f'   Model used: {_MODEL_LABEL}')

if not sc_files:
    print('⚠️  WARNING: no soundscape embeddings found — Cell 5 may have failed')

In [ ]:
# === CELL 7: UPLOAD TO KAGGLE AS birdclef-2026-perch-embs-v3 ===
import json

KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', 'chiragggg')

meta = {
    'title': 'birdclef-2026-perch-embs-v3',
    'id': f'{KAGGLE_USERNAME}/birdclef-2026-perch-embs-v3',
    'licenses': [{'name': 'CC0-1.0'}],
    'notes': f'Embeddings computed with {_MODEL_LABEL}. Focal + soundscape windows.',
}
with open(EMBD_DIR / 'dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

total = len(list(EMBD_DIR.glob('*.npy')))
print(f'Uploading {total} embeddings as {meta["id"]} ...')
os.system(f'kaggle datasets create -p {EMBD_DIR} --dir-mode zip')
print('Done!')
print('Next: run birdclef2026-train-v22-perch-retrain.ipynb')
print(f'  Add birdclef-2026-perch-embs-v3 as input (EMBD_DIR pointing to this dataset)')